| Company        | Ticker |
| -------------- | ------ |
| Microsoft      | MSFT   |
| Apple          | AAPL   |
| Amazon         | AMZN   |
| Tesla, Inc.    | TSLA   |
| NVIDIA         | NVDA   |

In [ ]:
import pandas as pd

In [128]:
news_df = pd.read_csv("../dataset_builder/dataset/final_news_dataset.csv")

In [ ]:
news_df.head()

,date,news
0,2008-08-08,Georgia 'downs two Russian warplanes' as count...
1,2008-08-08,Georgian troops retreat from S. Osettain capit...
2,2008-08-08,Indian shoe manufactory - And again in a seri...
3,2008-08-08,The 'enemy combatent' trials are nothing but a...
4,2008-08-08,This is a busy day: The European Union has ap...


In [130]:
# Filter Microsoft-related news
keywords = ["nvidia", "nvda"]

def is_msft(text):
    if pd.isna(text):
        return False
    text = text.lower()
    return any(k in text for k in keywords)

news_df = news_df[news_df['news'].apply(is_msft)]

In [131]:
news_df

,date,news
56284,2015-12-16,"MKM Partners Top Picks For 2016: ACE, CTXS, EA..."
63703,2018-02-08,"ICYMI: Snap Earnings, VIX Instruments, AMD And..."
63716,2018-02-08,"ICYMI: Nvidia, Snap, The VIX, And MoviePass"
63916,2018-03-12,Stocks That Made New 52-Wk Highs This Morning ...
64310,2018-05-04,"Morgan Stanley Previews Semi Earnings, Expects..."
...,...,...
120400,2026-04-25,Nvidia market cap hits $5T amid US-China tech ...
120401,2026-04-25,Nvidia poised to become largest company by mar...
120483,2026-04-26,Nvidia eyes top market cap spot amid tech earn...
120659,2026-04-28,"OpenAI Investors—Nvidia, Oracle, More—Fall Aft..."


In [132]:
news_df.drop_duplicates(inplace=True)

In [133]:
news_df.dtypes

date    str
news    str
dtype: object

In [134]:
import yfinance as yf

data = yf.download("NVDA", start="2008-01-01")

data.to_csv("nvda.csv")

[*********************100%***********************]  1 of 1 completed


In [135]:
ohlcv_df = pd.read_csv('nvda.csv')

In [136]:
ohlcv_df = ohlcv_df.dropna()
ohlcv_df = ohlcv_df.drop_duplicates()

In [137]:
# reset index so Date becomes a column
ohlcv_df = ohlcv_df.drop(0)   # removes row with index 0,1
ohlcv_df = ohlcv_df.reset_index(drop=True)
ohlcv_df = ohlcv_df.rename(columns={"Price": "date",})
# rename columns
ohlcv_df.head()

,date,Close,High,Low,Open,Volume
0,2008-01-02,0.7565767765045166,0.7849970707728766,0.7462629287476974,0.7820175171828906,483964000
1,2008-01-03,0.7506176233291626,0.7760583624486117,0.7478672495239742,0.7609314158093473,475308000
2,2008-01-04,0.6875887513160706,0.7318236455741811,0.6830048306788683,0.7281564981354904,736092000
3,2008-01-07,0.616537868976593,0.6979025346236167,0.6039320610841226,0.6921726070421376,1006800000
4,2008-01-08,0.6296021342277527,0.6713158364734526,0.6055365220662898,0.619975878434238,1106956000


In [138]:
ohlcv_df.dtypes

date      str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

In [139]:
cols = ohlcv_df.columns.drop(["date"])
ohlcv_df[cols] = ohlcv_df[cols].astype(float)

In [140]:
ohlcv_df["date"] = pd.to_datetime(ohlcv_df["date"])

In [141]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

Loading weights: 100%|███████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 22235.40it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [142]:
label_map = {
    "positive": 1,
    "negative": -1,
    "neutral": 0
}

def get_sentiment(text):
    try:
        result = sentiment_pipeline(str(text[:512]))[0]
        label = result['label'].lower()
        return label_map.get(label, 0)
    except:
        return 0

In [143]:
from tqdm import tqdm
tqdm.pandas()

news_df["sentiment"] = news_df["news"].progress_apply(get_sentiment)

100%|█████████████████████████████████████████████████████████████████████████████| 3086/3086 [05:22<00:00,  9.57it/s]


In [144]:
news_df

,date,news,sentiment
56284,2015-12-16,"MKM Partners Top Picks For 2016: ACE, CTXS, EA...",0
63703,2018-02-08,"ICYMI: Snap Earnings, VIX Instruments, AMD And...",0
63716,2018-02-08,"ICYMI: Nvidia, Snap, The VIX, And MoviePass",0
63916,2018-03-12,Stocks That Made New 52-Wk Highs This Morning ...,0
64310,2018-05-04,"Morgan Stanley Previews Semi Earnings, Expects...",1
...,...,...,...
120400,2026-04-25,Nvidia market cap hits $5T amid US-China tech ...,-1
120401,2026-04-25,Nvidia poised to become largest company by mar...,1
120483,2026-04-26,Nvidia eyes top market cap spot amid tech earn...,1
120659,2026-04-28,"OpenAI Investors—Nvidia, Oracle, More—Fall Aft...",-1


In [145]:
daily_sentiment = news_df.groupby("date").agg(
    sentiment_mean=("sentiment", "mean"),
    # news_count=("sentiment", "count")
).reset_index()

# Normalize sentiment
daily_sentiment["sentiment_mean"] = daily_sentiment["sentiment_mean"].fillna(0)

In [146]:
daily_sentiment

,date,sentiment_mean
0,2013-06-20,0.0
1,2015-01-27,-1.0
2,2015-04-04,1.0
3,2015-09-01,0.0
4,2015-12-16,0.0
...,...,...
1030,2026-04-20,0.0
1031,2026-04-22,0.0
1032,2026-04-25,0.0
1033,2026-04-26,1.0


In [147]:
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"])

In [148]:
final_df = pd.merge(
    ohlcv_df,
    daily_sentiment,
    on="date",
    how="left"
)

# Fill missing sentiment
final_df["sentiment_mean"] = final_df["sentiment_mean"].fillna(0)

In [149]:
final_df

,date,Close,High,Low,Open,Volume,sentiment_mean
0,2008-01-02,0.756577,0.784997,0.746263,0.782018,4.839640e+08,0.0
1,2008-01-03,0.750618,0.776058,0.747867,0.760931,4.753080e+08,0.0
2,2008-01-04,0.687589,0.731824,0.683005,0.728156,7.360920e+08,0.0
3,2008-01-07,0.616538,0.697903,0.603932,0.692173,1.006800e+09,0.0
4,2008-01-08,0.629602,0.671316,0.605537,0.619976,1.106956e+09,0.0
...,...,...,...,...,...,...,...
4605,2026-04-23,199.639999,203.830002,197.220001,202.460007,1.135618e+08,0.0
4606,2026-04-24,208.270004,210.949997,199.809998,199.960007,2.141344e+08,0.0
4607,2026-04-27,216.610001,216.830002,207.380005,209.649994,1.871724e+08,0.0
4608,2026-04-28,213.169998,214.729996,208.199997,209.490005,1.802754e+08,-1.0


In [150]:
final_df["sentiment_3d"] = final_df["sentiment_mean"].rolling(3).mean().fillna(0)
final_df["sentiment_7d"] = final_df["sentiment_mean"].rolling(7).mean().fillna(0)
final_df["sentiment_14d"] = final_df["sentiment_mean"].rolling(14).mean().fillna(0)

In [151]:
final_df.dtypes

date              datetime64[us]
Close                    float64
High                     float64
Low                      float64
Open                     float64
Volume                   float64
sentiment_mean           float64
sentiment_3d             float64
sentiment_7d             float64
sentiment_14d            float64
dtype: object

In [152]:
final_df.to_csv("nvda_sentiment.csv", index=False)